# Find distinct spectra with a convex hull

A spectral image can contain many pixels with similar mixtures. Lumic uses
a convex hull to find the spectra on the outer boundary of a reduced spectral
space. These pixels provide a smaller pool of candidates for estimating component
spectra.

This example uses three synthetic spectra and 400 mixtures, with three pure
pixels buried among them. It runs offline with Python 3.12 and
`lumic[examples]==0.1.0rc2` installed.

## Simulate pixel spectra

Each row of `pixels` is one pixel and each column is a detector channel. The
three reference spectra sum to one across wavelengths. Vary the expected
counts across pixels to show how the projection handles brightness differences.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from lumic import essential_spectral_pixles, spectral_uniqueness_map

rng = np.random.default_rng(42)
wavelengths = np.linspace(420, 700, 32)
peaks = np.array([460, 525, 610])
widths = np.array([20, 25, 30])
reference_spectra = np.exp(-0.5 * ((wavelengths - peaks[:, None]) / widths[:, None]) ** 2)
reference_spectra /= reference_spectra.sum(axis=1, keepdims=True)

fractions = np.vstack((rng.dirichlet(np.ones(3), size=400), np.eye(3)))
rng.shuffle(fractions)
counts = rng.uniform(200, 800, size=len(fractions))
pixels = (fractions @ reference_spectra) * counts[:, None]

## Find the hull vertices

Lumic projects the spectra with truncated singular value decomposition (SVD).
It divides each pixel's projected coordinates by the first coordinate to cancel
multiplicative brightness differences in that projection. With `npc=3`, two
coordinates remain for the convex hull.

The function returns both the selected pixel spectra in detector units and their
indices in the original pixel matrix.

In [ ]:
candidates, indices = essential_spectral_pixles(pixels, npc=3)

print(f"Selected {len(indices)} candidate spectra from {len(pixels)} pixels")
print("Candidate mixture fractions (simulation ground truth):")
print(fractions[indices])

## Plot the candidates

Use `spectral_uniqueness_map` to access the same kind of normalized SVD
coordinates. Here we pass the pixels as a one-row spectral image. The left
panel shows the candidate indices in that space. The right panel compares
their spectral shapes with the known references after scaling each to sum to one.
The simulation fractions let us label the candidates by component.

In [ ]:
projection = spectral_uniqueness_map(pixels[None, :, :], npc=3, spectral_axis=-1)
coordinates = projection.coordinates[0]
boundary = coordinates[np.r_[indices, indices[0]]]
candidate_shapes = candidates / candidates.sum(axis=1, keepdims=True)
component_indices = fractions[indices].argmax(axis=1)

fig, (ax_hull, ax_spectra) = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
ax_hull.scatter(*coordinates.T, s=12, alpha=0.45, color="0.45", label="Observed pixels")
ax_hull.plot(*boundary.T, color="0.3", linewidth=1, label="Convex hull")
for pixel_index, shape, component in zip(indices, candidate_shapes, component_indices):
    color = f"C{component}"
    ax_hull.scatter(*coordinates[pixel_index], s=65, color=color, zorder=3)
    ax_spectra.plot(wavelengths, shape, color=color, label=f"Candidate {component + 1}")
    ax_spectra.plot(wavelengths, reference_spectra[component], ":", color="black")

ax_hull.set(xlabel="Spectral coordinate 1", ylabel="Spectral coordinate 2",
            title="Candidates on the convex hull")
ax_hull.legend()
ax_spectra.plot([], [], ":", color="black", label="Known references")
ax_spectra.set(xlabel="Wavelength (nm)", ylabel="Fraction of total signal",
               title="Candidate spectral shapes")
ax_spectra.legend()
plt.show()

In this noiseless simulation the hull finds the three pure pixels, one at each
vertex. In experimental data, a hull vertex marks an extreme spectrum in the
chosen projection. It may be a mixture, a background spectrum, or a noisy
pixel. Use the hull as a candidate pool. For an example that selects
starting spectra from that pool and refines them with MCR-ALS, see
[Estimate unknown spectra](../../../docs/backend/estimate-unknown-spectra.md).